# 02 - Evidence Gate 0A: Route Intent

This notebook translates the Phase 1 deck and project clarification into route families.

It does not select a best route. It defines what route options must be tested later.

## Design Intent

The deck sequence is:

```text
City Core -> Transition -> Pavilion Node -> Knowledge Quarter
```

The correct deck identifies three deck-core Phase 1 route vectors: Colmore Row, Moor Street, and Nechells via Dartmouth Middleway. Project clarification also adds New Street and Snow Hill as experimental tactical-corridor origins. Therefore, the first route analysis should test the three deck-core route vectors plus the two experimental station corridors into the Ryder Street pavilion search area, then onward continuity into B-KQ.

In [ ]:
from pathlib import Path
import sys

root_candidates = (Path.cwd(), Path.cwd() / 'phase1_spinelens_ai', *Path.cwd().parents)
PROJECT_ROOT = next(path for path in root_candidates if (path / 'src' / 'spinelens').exists())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA = PROJECT_ROOT / 'data'
PROJECT_ROOT

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

nodes = pd.read_csv(DATA / 'route_nodes_phase1.csv')
families = pd.read_csv(DATA / 'route_families_phase1.csv')

nodes

In [ ]:
required_node_types = {'origin', 'gateway', 'destination'}
assert required_node_types <= set(nodes['node_type']), 'Route nodes must include origins, a gateway, and destinations.'
assert nodes['node_id'].is_unique, 'Route node IDs must be unique.'
assert (nodes['validation_status'] == 'not_validated').all(), 'Gate 0A nodes should remain unvalidated until reviewed.'

gateway = nodes.loc[nodes['route_role'] == 'primary_phase1_pavilion_site_search_area']
assert len(gateway) == 1, 'There must be exactly one primary Phase 1 gateway.'
gateway[['node_id', 'node_name', 'route_role', 'validation_action', 'notes']]

## Route Families

Route families are hypotheses to test, not findings. Each family should later be evaluated for legibility, comfort, crossing burden, night-time readability, and fit with the Phase 1 budget.

In [ ]:
known_nodes = set(nodes['node_id'])
for column in ['origin_node_id', 'gateway_node_id']:
    unknown = sorted(set(families[column]) - known_nodes)
    assert not unknown, f'Unknown node IDs in {column}: {unknown}'

onward = set(families['onward_anchor_id'].dropna()) - {''}
unknown_onward = sorted(onward - known_nodes)
assert not unknown_onward, f'Unknown onward anchors: {unknown_onward}'

families

## Straight-Line Sanity Check

These distances are not walking-route results. They only check whether the route families are geographically plausible before public network data is acquired.

In [ ]:
node_gdf = gpd.GeoDataFrame(
    nodes,
    geometry=[Point(lon, lat) for lon, lat in zip(nodes['longitude'], nodes['latitude'])],
    crs='EPSG:4326',
).to_crs('EPSG:27700')

geom_by_id = dict(zip(node_gdf['node_id'], node_gdf.geometry))

distance_rows = []
for row in families.itertuples(index=False):
    origin = geom_by_id[row.origin_node_id]
    gateway_geom = geom_by_id[row.gateway_node_id]
    onward_geom = geom_by_id.get(row.onward_anchor_id)
    distance_rows.append({
        'route_family_id': row.route_family_id,
        'origin_to_gateway_m': round(origin.distance(gateway_geom), 1),
        'gateway_to_onward_anchor_m': round(gateway_geom.distance(onward_geom), 1) if onward_geom is not None else None,
        'priority': row.priority,
        'status': row.status,
    })

pd.DataFrame(distance_rows)

## Gate 0A Decision

Before route modelling, reviewers must confirm:

- The exact Ryder Street grassland / pavilion search area.
- The exact Colmore Row, Moor Street, and Nechells/Dartmouth origin points.
- The exact New Street and Snow Hill exits for experimental tactical-corridor testing.
- Whether BCU/Eastside anchors should be added before the first network pull.

After that, Priority 1 source reachability checks can start.